# Diffusion Posterior Sampling (DPS) — Implementation / 구현

**Paper**: Chung, H., Kim, J., McCann, M. T., Klasky, M. L., & Ye, J. C. (2023). *ICLR 2023*. arXiv:2209.14687.

## Overview / 개요

이 노트북은 Chung et al. (2023)의 DPS 알고리즘 핵심 아이디어를 작은 토이 모델로 재구현한다.
This notebook reproduces the core ideas of DPS with a tiny toy model:

1. **Tweedie posterior mean** — `\hat x_0 = (x_t + (1-bar_alpha_t) s_theta(x_t,t)) / sqrt(bar_alpha_t)`.
2. **Score for a tractable Gaussian prior** — exact closed form `s(x) = -x` (so we don't need to train a real diffusion model).
3. **Linear inverse problem** — 32-pixel 1D signal downsampled by 2× averaging.
4. **DPS sampling loop** — backprop through Tweedie estimate to push iterate toward measurement consistency.

The key concept demonstrated: the gradient `\nabla_{x_t} || y - A * \hat x_0(x_t) ||^2` is computed via PyTorch autograd through the Tweedie estimate, exactly as in DPS.

학습된 거대 score 모델 대신 **가우시안 prior**를 사용 — 이때 Tweedie의 score가 닫힌 형식 `-x`로 알려져 있어 실제 학습 없이도 알고리즘 골격을 정확히 재현 가능.

> **Note**: this is a *concept-level* reproduction. Real DPS uses a U-Net score net trained on FFHQ/ImageNet, sampled at 256x256 with 1000 steps.


In [ ]:
from __future__ import annotations

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True

DEVICE = torch.device('cpu')   # tiny problem, CPU is fine

# Problem dimensions / 문제 크기
N: int = 32          # signal length
DS: int = 2          # downsampling factor (so y has N/DS = 16 measurements)
NOISE_STD: float = 0.05


---

## Part 1: Toy data and forward operator / 토이 데이터와 전방 연산자

We construct a synthetic 1D piecewise-constant signal `x_0` (analogue of an image row), and a forward operator `A` that downsamples by 2 (block-average pooling). Measurement `y = A x_0 + n` with Gaussian noise.

1D piecewise-constant 신호 `x_0` 위에 2× 다운샘플링 + 가우시안 잡음을 적용해 `y`를 생성. `A`는 32 → 16 차원 평균 풀링 행렬.


In [ ]:
def make_signal(n: int = N) -> torch.Tensor:
    """Return a piecewise-constant 1D signal in R^n.

    Args:
        n: Signal length.

    Returns:
        Signal tensor of shape (n,).
    """
    x = torch.zeros(n)
    x[0:8] = 1.0
    x[8:16] = -0.5
    x[16:24] = 0.7
    x[24:32] = -1.0
    return x


def make_downsample_matrix(n: int, ds: int) -> torch.Tensor:
    """Build A in R^{(n/ds) x n} that performs ds-fold block averaging."""
    m = n // ds
    A = torch.zeros(m, n)
    for i in range(m):
        A[i, ds * i:ds * (i + 1)] = 1.0 / ds
    return A


x_true = make_signal(N)
A = make_downsample_matrix(N, DS)

noise = NOISE_STD * torch.randn(A.shape[0])
y = A @ x_true + noise

print(f"x_true shape: {x_true.shape}, y shape: {y.shape}")
print(f"y first 4 values: {y[:4].numpy()}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(x_true, '-o', markersize=4); ax[0].set_title('Ground truth $x_0$')
ax[1].plot(y, '-s', color='C1', markersize=5); ax[1].set_title('Measurement $y$ (noisy 2x downsample)')
plt.tight_layout(); plt.show()


---

## Part 2: Diffusion noise schedule and Tweedie formula / 확산 노이즈 스케줄과 Tweedie 공식

For a tractable Gaussian prior `p(x_0) = N(0, I)`, the score has closed form
`\nabla log p_t(x_t) = -x_t / (bar_alpha_t * 1 + (1-bar_alpha_t) * 1) = -x_t`.

Standard DDPM cosine schedule with T=200. Tweedie estimate
`\hat x_0 = (x_t + (1 - bar_alpha_t) * s_theta(x_t, t)) / sqrt(bar_alpha_t)`
becomes `(x_t - (1 - bar_alpha_t) * x_t) / sqrt(bar_alpha_t) = sqrt(bar_alpha_t) * x_t` for our Gaussian prior.

가우시안 prior 가정 시 학습된 score net 없이 score = `-x`. 이때 Tweedie는 `\hat x_0 = sqrt(bar_alpha_t) * x_t`. 알고리즘 검증에 충분.


In [ ]:
T_STEPS: int = 200

def cosine_schedule(T: int = T_STEPS, s: float = 0.008) -> torch.Tensor:
    """DDPM cosine schedule for bar_alpha_t (Nichol & Dhariwal 2021)."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f = torch.cos(((steps / T) + s) / (1 + s) * math.pi / 2) ** 2
    bar_alpha = f / f[0]
    return bar_alpha[1:].float()  # length T


bar_alpha = cosine_schedule(T_STEPS)
alpha = bar_alpha / torch.cat([torch.tensor([1.0]), bar_alpha[:-1]])

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(bar_alpha); ax[0].set_title('$\\bar\\alpha_t$ (cosine schedule)')
ax[0].set_xlabel('t')
ax[1].plot(torch.sqrt(1.0 - bar_alpha)); ax[1].set_title('$\\sigma_t = \\sqrt{1 - \\bar\\alpha_t}$')
ax[1].set_xlabel('t')
plt.tight_layout(); plt.show()


def gaussian_prior_score(x_t: torch.Tensor, t: int) -> torch.Tensor:
    """Closed-form score for p(x_0) = N(0, I) under VP-SDE.

    For x_t = sqrt(bar_alpha_t) x_0 + sqrt(1 - bar_alpha_t) eps with x_0 ~ N(0, I),
    x_t ~ N(0, I), so score = -x_t.

    Args:
        x_t: Noisy iterate.
        t: Current timestep (unused; signature kept for general compatibility).

    Returns:
        Score (gradient of log density) at x_t.
    """
    return -x_t


def tweedie_x0_hat(x_t: torch.Tensor, t: int) -> torch.Tensor:
    """Tweedie posterior-mean estimate of x_0 given x_t."""
    ba = bar_alpha[t]
    s = gaussian_prior_score(x_t, t)
    return (x_t + (1.0 - ba) * s) / torch.sqrt(ba)


---

## Part 3: DPS sampling loop / DPS 샘플링 루프

Algorithm 1 of the paper, simplified.  At each step we
1. Compute Tweedie estimate `\hat x_0` *with grad* (set `x_t.requires_grad_`).
2. Compute measurement loss `|| y - A \hat x_0 ||^2`.
3. Backprop to get `\nabla_{x_t} loss`.
4. Take a standard DDPM ancestral step on `\hat x_0`.
5. Subtract `zeta * grad` (DPS likelihood guidance).

논문 Algorithm 1의 단순화 — autograd로 measurement loss의 `x_t`-방향 기울기를 얻고, ancestral DDPM step에 추가.


In [ ]:
def dps_sample(
    A: torch.Tensor,
    y: torch.Tensor,
    n: int = N,
    T: int = T_STEPS,
    sigma_y: float = NOISE_STD,
    zeta_prime: float = 1.0,
    return_history: bool = False,
):
    """Run DPS reverse process for the given linear forward operator.

    Args:
        A: Forward operator (m x n).
        y: Measurement vector (m,).
        n: Signal length.
        T: Number of reverse steps.
        sigma_y: Measurement noise standard deviation.
        zeta_prime: DPS step size before normalisation.
        return_history: If True, return the (x_t, x_hat0) trajectory.

    Returns:
        Final sample (n,) and optionally the history list.
    """
    x_t = torch.randn(n)
    history = []
    for t in reversed(range(T)):
        x_t = x_t.detach().clone().requires_grad_(True)
        x0_hat = tweedie_x0_hat(x_t, t)
        residual = y - A @ x0_hat
        loss = (residual ** 2).sum()
        grad = torch.autograd.grad(loss, x_t)[0]

        # Norm-normalised step size (Eq. 17)
        zeta_t = zeta_prime / (residual.norm() + 1e-8)

        # Ancestral DDPM step on x0_hat (DDIM-style with eta=1)
        with torch.no_grad():
            ba_t = bar_alpha[t]
            ba_prev = bar_alpha[t - 1] if t > 0 else torch.tensor(1.0)
            sigma_step = ((1 - ba_prev) / (1 - ba_t) * (1 - ba_t / ba_prev)).clamp_min(0).sqrt()
            mean = ba_prev.sqrt() * x0_hat + (1 - ba_prev - sigma_step ** 2).clamp_min(0).sqrt() * (-x_t * (1 - ba_t).sqrt() / (1 - ba_t).sqrt())
            # Alternatively (equivalent for Gaussian prior):
            # mean = ba_prev.sqrt() * x0_hat + sqrt(1 - ba_prev - sigma^2) * predicted_eps
            x_prev = mean + sigma_step * torch.randn(n) if t > 0 else mean

        # DPS guidance subtraction
        x_t = (x_prev - zeta_t * grad).detach()

        if return_history and t % 20 == 0:
            history.append((t, x_t.clone(), x0_hat.detach().clone()))

    return (x_t.detach(), history) if return_history else x_t.detach()


x_dps, hist = dps_sample(A, y, return_history=True)
print(f"Sampled x shape: {x_dps.shape}, MSE vs truth: {((x_dps - x_true) ** 2).mean().item():.4f}")


---

## Part 4: Compare with baselines / 기준선 비교

To understand DPS's value, compare with three baselines:
1. **Naive zero-fill** — upsample `y` to `n` by repetition (no prior, no fidelity adjustment).
2. **Least-squares pseudoinverse** — `A^+ y`. Optimal in fidelity but no prior, noise-amplifying when `A` is ill-conditioned.
3. **MAP with Gaussian prior** — `(A^T A / sigma_y^2 + I)^{-1} A^T y / sigma_y^2`. Closed form for our toy model.

세 가지 비교: 단순 업샘플, 최소제곱, MAP closed-form. DPS는 generative posterior sample을 제공.


In [ ]:
def upsample_repeat(y: torch.Tensor, ds: int, n: int) -> torch.Tensor:
    return y.repeat_interleave(ds)[:n]

def map_gaussian(A: torch.Tensor, y: torch.Tensor, sigma_y: float, n: int) -> torch.Tensor:
    H = A.T @ A / (sigma_y ** 2) + torch.eye(n)
    return torch.linalg.solve(H, A.T @ y / (sigma_y ** 2))


x_zero = upsample_repeat(y, DS, N)
x_pinv = torch.linalg.pinv(A) @ y
x_map = map_gaussian(A, y, NOISE_STD, N)

mse = lambda a, b: ((a - b) ** 2).mean().item()
print(f"MSE  zero-fill = {mse(x_zero, x_true):.4f}")
print(f"MSE  pseudoinv = {mse(x_pinv, x_true):.4f}")
print(f"MSE  MAP-Gauss = {mse(x_map, x_true):.4f}")
print(f"MSE  DPS       = {mse(x_dps, x_true):.4f}")

fig, ax = plt.subplots(1, 1, figsize=(11, 4))
ax.plot(x_true, '-', linewidth=2, label='ground truth', alpha=0.7)
ax.plot(x_zero, ':', label='zero-fill upsample')
ax.plot(x_pinv, '-.', label='pseudoinverse')
ax.plot(x_map, '--', label='MAP (Gaussian)')
ax.plot(x_dps, '-o', markersize=3, label='DPS', color='red')
ax.legend(); ax.set_title('DPS vs baselines on 1D 2x downsampling')
plt.tight_layout(); plt.show()


---

## Part 5: Visualise the Tweedie trajectory / Tweedie 궤적 시각화

Plot `\hat x_0` at several timesteps. Early in the reverse process, `\hat x_0` is noisy/inaccurate; as `t -> 0`, it converges to a measurement-consistent sample.

샘플링 도중 다양한 `t`에서의 Tweedie 추정값을 그려본다. 초반에는 noise-dominated, 후반에는 측정 일관 신호.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharey=True)
axes = axes.ravel()

# pick six checkpoints from history (every other entry)
checkpoints = hist[::max(1, len(hist) // 6)][:6]
for ax, (t, x_t_chk, x0_hat_chk) in zip(axes, checkpoints):
    ax.plot(x_true, 'k-', alpha=0.4, label='truth')
    ax.plot(x0_hat_chk, '-r', label='$\\hat x_0$ (Tweedie)')
    ax.plot(x_t_chk, ':b', alpha=0.6, label='$x_t$')
    ax.set_title(f't = {t}')
    ax.legend(fontsize=8)

fig.suptitle('Tweedie estimate $\\hat x_0$ along the DPS trajectory')
plt.tight_layout(); plt.show()


---

## Part 6: Effect of step size zeta / 스텝 크기 zeta의 영향

The DPS step size `zeta` controls the strength of measurement guidance. Too small → ignores measurement; too large → overshoots, leaves data manifold.

Sweep `zeta_prime in {0.1, 0.5, 1.0, 5.0, 20.0}` and report MSE.

`zeta`가 너무 작으면 측정값 무시, 너무 크면 prior 이탈. norm-normalised 스케일에서 적정값 탐색.


In [ ]:
zetas = [0.1, 0.5, 1.0, 5.0, 20.0]
mses = []
for z in zetas:
    torch.manual_seed(0)  # same noise sample for fair comparison
    x_dps_z = dps_sample(A, y, zeta_prime=z)
    mses.append(mse(x_dps_z, x_true))

print('zeta   MSE')
for z, m in zip(zetas, mses):
    print(f'{z:5.2f}  {m:.4f}')

fig, ax = plt.subplots()
ax.plot(zetas, mses, '-o')
ax.set_xscale('log'); ax.set_xlabel("$\\zeta'$"); ax.set_ylabel('MSE vs truth')
ax.set_title('DPS quality vs measurement-guidance step size')
plt.tight_layout(); plt.show()


---

## Part 7: Summary / 요약

### What we demonstrated / 구현으로 보여준 것
- The **Tweedie posterior mean** `\hat x_0 = (x_t + (1-bar_alpha_t) s_theta) / sqrt(bar_alpha_t)` is a closed-form estimate computable from the score.
- The **DPS likelihood gradient** `\nabla_{x_t} || y - A \hat x_0 ||^2` is computed via PyTorch autograd by setting `x_t.requires_grad_` before the Tweedie computation.
- The **norm-normalised step size** `zeta_t = zeta_prime / || y - A \hat x_0 ||` provides robustness across measurement scales.
- DPS provides a measurement-consistent sample without an explicit projection step (unlike MCG).

### What we did NOT do / 하지 않은 것
- We did NOT train a real diffusion model — used closed-form Gaussian-prior score.
- We did NOT test on real images — used a 32-pixel 1D toy.
- We did NOT compare with DDRM, MCG, or PnP-ADMM directly.

### To do for a full reproduction / 전체 재현을 위한 추가 작업
- Replace `gaussian_prior_score` with a U-Net trained on FFHQ.
- Extend `A` to a 2D convolution / Fourier mask / nonlinear operator.
- Add Poisson NLL variant for low-photon imaging applications.

### Real-world relevance / 실세계 적용
DPS의 핵심 가치는 동일한 사전학습 score 모델을 *어떤 미분 가능한 forward model*에도 재사용 가능하다는 점. 천체관측, MRI, CT 등 forward model이 task별로 다른 imaging 분야에서 강력.

### References
- Chung, H. et al. (2023). *Diffusion Posterior Sampling for General Noisy Inverse Problems.* ICLR.
- Code: https://github.com/DPS2022/diffusion-posterior-sampling
